# Path dependency and Monte Carlo simulations: code development

Options with payoffs that depend on the path taken by the underlying and not only on the value at expiration are said to be path-dependent. In this notebook, we will analyze Monte Carlo Simulation of these options. 

- Options analyzed in this notebook do not have closed-form solutions
- We classify options as weakly or strongly path dependent
- Path dependency is manifest as an extra variable in the option pricing besides $t$ and $S_t$ characterizing the path


In [1]:
import sys
sys.path.append("../src")

import numpy as np

## Weak path dependency 

### American put
American options incorporate early exercise and path dependence but we can still characterize the price by S_t and t. However early excercise makes the American options an optimization problem. We will model it following the classic paper by Longstaff-Schwartz  "_Valuing American Options by Simulation: A Simple Least-Squares Approach_", Review of Financial Studies 14(1), 113-147.  



Let us consider the following paths for an American put with strike price $E=1.10$ and  riskless rate $r=0.06$

||||||
| :-: | :-: | :-: | :-: | :-: |
| Path | $t=0$ | $t=1$ | $t=2$ | $t=3$ |
| 1 | 1.00 | 1.09 | 1.08 | 1.34|
| 2 | 1.00 | 1.16 | 1.26 | 1.54|
| 3 | 1.00 | 1.22 | 1.07 | 1.03|
| 4 | 1.00 | 0.93 | 0.97 | 0.92|
| 5 | 1.00 | 1.11 | 1.56 | 1.52|
| 6 | 1.00 | 0.76 | 0.77 | 0.90|
| 7 | 1.00 | 0.92 | 0.84 | 1.01|
| 8 | 1.00 | 0.88 | 1.22 | 1.34|

Starting at final time $t=3$ we would exercise only for the paths $3,4,6,7$


In [114]:
E=1.1
rate=0.06
T=3
n_steps=3
dt=T/n_steps

#### Situation at  $t_3$

In [115]:
paths=np.array([[ 1.00 , 1.09 , 1.08 , 1.34], [1.00 , 1.16 , 1.26 , 1.54], [1.00 , 1.22 , 1.07 , 1.03], [1.00 , 0.93 , 0.97 , 0.92],
                [ 1.00 , 1.11 , 1.56 , 1.52], [ 1.00 , 0.76 , 0.77 , 0.90], [1.00 , 0.92 , 0.84 , 1.01], [ 1.00 , 0.88 , 1.22 , 1.34]
               ])

payoffs_t3= np.max(np.concatenate((E-(paths.T)[-1], np.zeros(8))).reshape(2,8), axis=0)
print("Payoffs are identical to the cashflows if the options were European")
print()
print(payoffs_t3)

Payoffs are identical to the cashflows if the options were European

[0.   0.   0.07 0.18 0.   0.2  0.09 0.  ]


Define a function that computes payoff

In [116]:
def payoff_put(paths, strike_price, m_paths, step):
    res= np.max(np.concatenate((strike_price-(paths.T)[-step], np.zeros(m_paths))).reshape(2,m_paths), axis=0)
    return res

print(payoff_put(paths, E, 8, 1))


[0.   0.   0.07 0.18 0.   0.2  0.09 0.  ]


The decision to exercise is trivial

In [117]:
decision_t3=np.zeros(8) 
pos_true_t3=np.array(payoffs_t3>decision_t3, dtype=int)
print(pos_true_t3)

[0 0 1 1 0 1 1 0]


In [118]:
cashflow_t3=payoffs_t3

#### Now let us look at $t_2$

In [119]:
payoffs_t2= payoff_put(paths, E, 8, 2) 
non_zero_tuple_t2=np.nonzero(payoffs_t2)

print('payoffs at t2 are ', payoffs_t2)


payoffs at t2 are  [0.02 0.   0.03 0.13 0.   0.33 0.26 0.  ]


Now we use these values to regress the discounted payoffs: $E[Y_{t_2}|X_{t_3}]]=a_0 X_{t_3}^2+a_1 X_{t_3} +a_2$

In [120]:
#Xs
paths_for_regression_t2=(paths.T[-2])[non_zero_tuple_t2]

payoffs_for_regression_t2= cashflow_t3[non_zero_tuple_t2]
#Ys
discounted_payoffs_t2= np.exp(-rate*dt)*payoffs_for_regression_t2

print(paths_for_regression_t2)
print(payoffs_for_regression_t2)
print(discounted_payoffs_t2)

[1.08 1.07 0.97 0.77 0.84]
[0.   0.07 0.18 0.2  0.09]
[0.         0.06592352 0.16951762 0.18835291 0.08475881]


We obtain the coefficients using polyfit

In [121]:
coeffs_t2=np.polyfit(paths_for_regression_t2,discounted_payoffs_t2,2)
#define function with fitted coefficients
def val_time(x, coeffs):
    return np.sum(coeffs*[x**2, x, 1])
print(coeffs_t2)

[-1.81357618  2.98341063 -1.06998766]


In [122]:
continuation_values_t2=np.array([val_time(x, coeffs_t2)  for x in paths_for_regression_t2])
print(payoffs_t2[non_zero_tuple_t2])

print(continuation_values_t2)

[0.02 0.03 0.13 0.33 0.26]
[0.03674056 0.04589834 0.11752682 0.15196921 0.15641792]


Decision making

In [123]:
decision_t2=np.array([val_time(x, coeffs_t2)*(val_time(x, coeffs_t2)>0)  for x in (paths.T[-2])])
pos_true_t2=np.array(payoffs_t2>decision_t2, dtype=int)
cashflow_t2=payoffs_t2*pos_true_t2

print(cashflow_t2)


[0.   0.   0.   0.13 0.   0.33 0.26 0.  ]


Cashflow must be updated

In [124]:
pos_true_t3=(pos_true_t3-pos_true_t2)
cashflow_t3=payoffs_t3*pos_true_t3
print(cashflow_t3)

[0.   0.   0.07 0.   0.   0.   0.   0.  ]


#### Situation at $t=1$

In [125]:
payoffs_t1= payoff_put(paths, E, 8,  3) 
non_zero_tuple_t1=np.nonzero(payoffs_t1)
print('payoffs at t1 are ', payoffs_t1)

payoffs at t1 are  [0.01 0.   0.   0.17 0.   0.34 0.18 0.22]


In [126]:
paths_for_regression_t1=(paths.T[-3])[non_zero_tuple_t1]
payoffs_for_regression_t1= cashflow_t2[non_zero_tuple_t1]
discounted_payoffs_t1= np.exp(-rate*dt)*payoffs_for_regression_t1

print(paths_for_regression_t1)
print(payoffs_for_regression_t1)
print(discounted_payoffs_t1)


[1.09 0.93 0.76 0.92 0.88]
[0.   0.13 0.33 0.26 0.  ]
[0.         0.12242939 0.3107823  0.24485878 0.        ]


In [127]:
coeffs_t1=np.polyfit(paths_for_regression_t1, discounted_payoffs_t1,2)
print(coeffs_t1)

[ 1.35645659 -3.3354434   2.03751234]


In [128]:
continuation_values_t1=np.array([val_time(x, coeffs_t1)  for x in paths_for_regression_t1])
print(payoffs_t1[non_zero_tuple_t1])
print(continuation_values_t1)

[0.01 0.17 0.34 0.18 0.22]
[0.01348511 0.10874928 0.28606468 0.11700927 0.15276213]


Decision $t_1$

In [130]:
decision_t1=np.array([val_time(x, coeffs_t1)*(val_time(x, coeffs_t1)>0)  for x in (paths.T[-3])])
pos_true_t1=np.array(payoffs_t1>decision_t1, dtype=int)
cashflow_t1=payoffs_t1*pos_true_t1
print(cashflow_t1)

[0.   0.   0.   0.17 0.   0.34 0.18 0.22]


In [131]:
#update pos_true
pos_true_t2=np.array((pos_true_t2-pos_true_t1)>0, dtype=int) 

In [132]:
cashflow_t2=payoffs_t2*pos_true_t2

In [133]:
pos_true=np.array([pos_true_t1, pos_true_t2, pos_true_t3])
print(pos_true.T)

[[0 0 0]
 [0 0 0]
 [0 0 1]
 [1 0 0]
 [0 0 0]
 [1 0 0]
 [1 0 0]
 [1 0 0]]


In [134]:
cashflow=np.array([cashflow_t1,cashflow_t2,cashflow_t3])
print(cashflow.T)

[[0.   0.   0.  ]
 [0.   0.   0.  ]
 [0.   0.   0.07]
 [0.17 0.   0.  ]
 [0.   0.   0.  ]
 [0.34 0.   0.  ]
 [0.18 0.   0.  ]
 [0.22 0.   0.  ]]


In [135]:
discounted_cashflow=[cashflow_t1*np.exp(-rate*dt),cashflow_t2*np.exp(-2*rate*dt),cashflow_t3*np.exp(-3*rate*dt)]
print(discounted_cashflow)

[array([0.        , 0.        , 0.        , 0.16009997, 0.        ,
       0.32019994, 0.16951762, 0.2071882 ]), array([0., 0., 0., 0., 0., 0., 0., 0.]), array([0.        , 0.        , 0.05846891, 0.        , 0.        ,
       0.        , 0.        , 0.        ])]


In [136]:
price=np.sum([np.mean(path) for path in discounted_cashflow])

print('The price of American Put is ', price)

The price of American Put is  0.11443433004505697


### Functions to implement above derivation

In [85]:
# Function to generates Geometric Brownian Paths
def generate_GBM_paths(S0=100, rate=0.06, volatility=0.5, m_paths=2, t_exercise=1, n_steps=252, seed=918723098):
    import numpy as np
    rng=np.random.default_rng(seed)
    dt=t_exercise/n_steps
    Ns=rng.standard_normal(size=(m_paths, n_steps))
    #Exact solution for GBM
    exp_factor=np.exp((rate-0.5*volatility**2)*dt+volatility*Ns*np.sqrt(dt))
    S_iterated=S0*np.cumprod(exp_factor,axis=1)
    S=np.concatenate([np.full((m_paths,1), S0), S_iterated], axis=1)
    return S

def payoff_put(paths, strike_price, m_paths, step):
    res= np.max(np.concatenate((strike_price-(paths.T)[-step], np.zeros(m_paths))).reshape(2,m_paths), axis=0)
    return res

In [220]:
def pricing_american_put(paths, m_paths=8, n_steps=3, t_exercise=3, strike_price=1.1): 
        dt=t_exercise/n_steps
        #intitialize auxiliary arrays
        S=paths
        payoffs=np.zeros_like(S)
        cashflows=np.zeros_like(S)
        pos_trues=np.zeros_like(S)
        payoffs[:,-1]= payoff_put(S, strike_price, m_paths, 1)
        #Initialize the cashflow 
        cashflows[:,-1]=payoffs.T[-1]
        pos_trues[:,-1]=np.array((payoffs.T)[-1]>0, dtype=int)
    
        #Assuming a power-like regression 
        def val_time(x, coeffs):
                return np.sum(coeffs*[x**2, x, 1])
        
        for n in range(n_steps-1):

            #Xs
            payoffs[:,-n-2] = payoff_put(S, strike_price, m_paths, n+2)
            current_payoff=payoffs.T[-n-2]    
            non_zero_tuple=np.nonzero(current_payoff)
            paths_for_regression=((S.T)[-n-2])[non_zero_tuple]
            #Ys
            cashflow_for_regression=cashflows.T[-n-1]                    
            payoffs_for_regression= cashflow_for_regression[non_zero_tuple]
            discounted_payoffs= np.exp(-rate*(n+1)*dt)*payoffs_for_regression
            
            regression= np.polyfit(paths_for_regression, discounted_payoffs,2)
             
            #Decision   making 
            decision=np.array([val_time(x, regression)*(val_time(x, regression)>0)  for x in (S.T[-n-2])])
            pos_trues[:,-n-2]=np.array(current_payoff>decision, dtype=int)
            cashflows[:,-n-2]=current_payoff*(pos_trues.T[-n-2])

            #The cashflow affects the last step
            pos_trues[:,-n-1]=np.array((pos_trues.T[-n-1]-pos_trues.T[-n-2])>0, dtype=int)          
            cashflows[:,-n-1]=cashflow_for_regression*(pos_trues.T[-n-1])
        #After recursion we discount the steps taken
        disval=rate
        discounted_cashflow=((np.exp(-disval))**(np.array(range(n_steps+1))))*cashflows
    #    discounted_cashflow=(np.exp(-disval))*cashflows
        #The price is the mean of each path
     
        price=np.sum([np.mean(path) for path in  discounted_cashflow.T]) 
        return price
    


In [221]:
pricing_american_put(paths)

np.float64(0.11443433004505697)

In [222]:
np.exp(-rate*3)*0.07

np.float64(0.058468914798789046)

In [223]:
m_test=1_000
S0_test=36
volatility_test=0.2
rate_test=0.06
t_test=1 
n_steps_test=50
strike_price_test=40

dt_test= t_test/n_steps_test
paths_test=generate_GBM_paths(S0=S0_test, rate=rate_test, volatility=volatility_test, m_paths=m_test, t_exercise=t_test, n_steps=n_steps_test, seed=1928309281)
price= pricing_american_put(paths_test, m_paths=m_test, t_exercise=t_test, n_steps=n_steps_test,  strike_price=strike_price_test)

In [224]:
price

np.float64(14.652535483206105)

In [209]:
price*np.exp(-rate)

np.float64(22.380886430616215)

In [186]:
np.mean((np.exp(-rate*n_steps_test)*(cashflow.T)[-1]))

np.float64(0.0036510516803100242)

In [373]:
import numpy as np

def longstaff_schwartz_put(S0, K, r, sigma, T, N, M, seed=42, deg=3):
    """Longstaff-Schwartz Monte Carlo for an American put."""
    rng = np.random.default_rng(seed)
    dt = T / N
    G  = rng.standard_normal(size=(M, N))
    # Vectorised path simulation: exact-GBM step is multiplicative, so
    # the entire price matrix is one np.cumprod call along time.
    a = np.exp((r - 0.5*sigma**2)*dt + sigma*np.sqrt(dt) * G)        # (M, N)
    S_inner = S0 * np.cumprod(a, axis=1)                              # (M, N)
    S = np.concatenate([np.full((M, 1), S0), S_inner], axis=1)        # (M, N+1)
    S= generate_GBM_paths(S0, r, sigma, M, T, N, seed) 
   # S=paths
    # Backward induction: this loop over n is *inherently sequential*
    # because the regression at time n needs the (already-updated) value
    # function V at time n+1. It cannot be replaced by a single numpy
    # call. Each step's internal work (regression, exercise decision,
    # value update) is fully vectorised over paths.
    V = np.maximum(K - S[:, -1], 0.0)
    exercise = np.zeros_like(S, dtype=bool)
    exercise[:, -1] = V > 0

    # Polynomial basis of degree `deg` in the underlying: [1, S, S^2, S^3]
    def design(s):
        return np.vstack([s**k for k in range(deg + 1)]).T

    for n in range(N - 1, 0, -1):
        V=V*np.exp(-r*dt)
        itm = np.where(S[:, n] < K)[0]   # ITM region for a put
        if len(itm) == 0:
            continue
        X = design(S[itm, n])
        Y = V[itm]
        #Y = np.exp(-r * dt * (N - n)) * V[itm]
        # Least-squares regression of (discounted) continuation value on basis
        coef, *_ = np.linalg.lstsq(X, Y, rcond=None)
        cont = X @ coef
        ex_now = np.maximum(K - S[itm, n], 0.0)
        ex_idx = itm[cont < ex_now]
        # Update value: exercise here, or carry forward discounted value
        V[ex_idx] = ex_now[cont < ex_now]
        exercise[ex_idx, :] = False
        exercise[ex_idx, n] = True
    
    price = np.exp(-r*dt)*V.mean()
   # price = np.exp(-r*0)*V.mean()
    return price

In [376]:
from scipy.stats import norm

def american_put_fdm(S0, K, r, sigma, T, Nx=500, Nt=5000, x_range=3.0):
    """American put via explicit finite-difference PDE method in log-space.

    Solves the Black-Scholes PDE in log-coordinates (x = log S, tau = T - t):
        dV/dtau = (r - 0.5*sigma^2) dV/dx + (0.5*sigma^2) d2V/dx2 - r*V
        V(tau=0, x) = max(K - exp(x), 0)   (terminal payoff)
        V(tau, x) >= max(K - exp(x), 0)     (early-exercise constraint)

    The log-space transform makes the PDE constant-coefficient, simplifying
    the discretisation and CFL stability condition. Upwind differencing
    is used for the convection term (r - 0.5*sigma^2) dV/dx to avoid
    oscillations. The early-exercise constraint is applied as a max()
    floor after each explicit Euler time step.
    """
    x0 = np.log(S0)
    x_min, x_max = x0 - x_range, x0 + x_range
    x_grid = np.linspace(x_min, x_max, Nx + 1)
    dx = x_grid[1] - x_grid[0]
    dt = T / Nt
    S_grid = np.exp(x_grid)
    V = np.maximum(K - S_grid, 0.0)       # terminal payoff
    mu_x = r - 0.5 * sigma**2             # log-space drift
    cfl = dt * 0.5 * sigma**2 / dx**2     # CFL stability number

    # Time-stepping loop is inherently sequential (each step depends on
    # the previous one). The spatial discretisation is vectorised.
    for n in range(Nt):
        dV_dx = np.zeros_like(V)
        # Upwind: backward difference if mu_x > 0, forward if mu_x < 0
        if mu_x > 0:
            dV_dx[1:] = (V[1:] - V[:-1]) / dx
        else:
            dV_dx[:-1] = (V[1:] - V[:-1]) / dx
        d2V_dx2 = np.zeros_like(V)
        d2V_dx2[1:-1] = (V[2:] - 2*V[1:-1] + V[:-2]) / dx**2
        L_V = mu_x * dV_dx + 0.5 * sigma**2 * d2V_dx2 - r * V
        V_new = V + dt * L_V              # explicit Euler step
        V_new = np.maximum(V_new, K - S_grid)  # early-exercise constraint
        V = V_new

    price = np.interp(np.log(S0), x_grid, V)
    return price, cfl

In [377]:
longstaff_schwartz_put(S0=S0_test, K=strike_price_test, r=rate, sigma=volatility_test, T=t_test, N=n_steps_test, M=m_test, seed=1928309281, deg=2)

np.float64(4.461698930990304)

In [432]:
american_put_fdm(S0=S0_test, K=strike_price_test, r=rate, sigma=volatility_test, T=t_test, Nx=500, Nt=50000, x_range=4.0)

(np.float64(4.470618368502852), np.float64(0.0015624999999999975))

In [ ]:

#define the parameters of the walk
T=1
volatility=0.5
n_steps=252
dt=T/n_steps
r=0.03
S0=120
strike=90

#number of paths
m_paths=1500

#the class for random number generation: suggested by numpy documentation
seed= 1028937912874


In [ ]:
price_american_put=longstaff_schwartz_put(S0, K=strike, r=r, sigma=volatility, T=T, N=n_steps, M=m_paths, seed=seed, deg=4)

In [ ]:
#compare against the vanilla put
from vanilla_options import bs_put

price_vanilla_put=bs_put(S0, K=strike, r=r, sigma=volatility, T=T)

In [ ]:
print(price_american_put)
print(price_vanilla_put)


## Strong path dependency 

### Floating-strike Asian call option
The payoff of this options reads

$$
G(\{S_t\})= \max(S_T-A_T, 0), \quad  A_T= \frac 1 N \sum_{k=1}^N S_{t_k} \, ,
$$
where  $A_T$ is the arithmetic average over the $N$ fixing dates $t_1, t_2, \dots, t_N$

In [ ]:
#We use the simulation of the GBM to smulate the paths

import numpy as np

#define the parameters of the walk
T=1
volatility=0.5
n_steps=252
dt=T/n_steps
r=0.03
S0=100

#number of paths
m_paths=150 

#the class for random number generation: suggested by numpy documentation
seed= 1028937912874
rng= np.random.default_rng(seed)


dW=rng.standard_normal(size=(m_paths, n_steps))*np.sqrt(dt)

#We use the exact solution for S_t 
#option 1. Adding the exponents
W=np.cumsum(dW,axis=1)
W=np.concatenate([np.zeros((m_paths,1)),W],axis=1)
t=np.linspace(0,T,n_steps+1)
S1=S0*np.exp((r-0.5*volatility**2)*t+volatility*W)

#option 2,as in the discretized approach
factors=np.exp((r-0.5*volatility**2)*dt+volatility*dW)

paths=S0*np.cumprod(factors, axis=1)
S2=np.concatenate([np.full((m_paths, 1), S0), paths],axis=1)

#Aritmethic average
A_T=S2.mean(axis=1) 

# pricing

payoff=np.maximum(S2[:,-1]-A_T, 0.0)
price= np.exp(-r*T)*payoff.mean()
error=np.exp(-r*T)*payoff.std(ddof=1)/np.sqrt(m_paths)


It is interesting to compare an Asian call option against a Vanilla call: Asians are usually cheaper

In [ ]:
from vanilla_options import bs_call

In [ ]:
price_bs_call=bs_call(S0, S0, r, volatility, T)

In [ ]:
print(f'The price of an Asian call is {price}')
print(f'the price of an vanilla call option (At the Money) is {price_bs_call}')


###  Floating-strike Asian put option with early exercise

Now, we will americanize the asian put introducing early exercise

In [ ]:
import numpy as np
from scipy.stats import norm

def lsm_american_asian_put(S0, K, r, sigma, T, N, M, seed=42):
    """Longstaff-Schwartz Monte Carlo for an American-style
    arithmetic-average Asian put.

    Payoff if exercised at time t_n: max(K - A_{t_n}, 0) where
    A_{t_n} = (1/(n+1)) * sum_{k=0..n} S_{t_k}.

    This is the canonical STRONGLY path-dependent option (Wilmott,
    *Paul Wilmott on Quantitative Finance*, Chapter 22): combining
    early exercise (which alone is weakly path-dependent via V(S, t))
    with arithmetic averaging (which alone is weakly path-dependent
    via V(S, t, A)) creates a problem whose LSM regression basis must
    include powers of BOTH S and A. The state space is (t, S, A) with
    a free-boundary surface in 2D --- qualitatively harder than either
    component alone.
    """
    rng = np.random.default_rng(seed)
    dt = T / N
    # Vectorised path simulation via cumprod
    G  = rng.standard_normal(size=(M, N))
    a  = np.exp((r - 0.5*sigma**2)*dt + sigma*np.sqrt(dt) * G)
    S  = np.concatenate([np.full((M, 1), S0), S0 * np.cumprod(a, axis=1)], axis=1)  # (M, N+1)

    # Running arithmetic average A_{t_n} = (1/(n+1)) * sum_{k=0..n} S_{t_k}
    # (includes t_0 = S_0 in the average)
    cumsum_S = np.cumsum(S, axis=1)
    A  = cumsum_S / np.arange(1, N + 2)             # (M, N+1)

    # Exercise payoff at each t_n: max(K - A_{t_n}, 0) for a put on the average
    exercise_payoff = np.maximum(K - A, 0.0)      # (M, N+1)

    # Backward induction: V starts as terminal payoff at t_N
    V = exercise_payoff[:, -1].copy()

    # 2D polynomial basis in (S, A) - includes interaction terms because
    # the early-exercise boundary depends on BOTH S and A jointly.
    def design(s, a):
        return np.vstack([
            np.ones_like(s), s, s**2, a, a**2, s*a
        ]).T

    # Backward induction loop is inherently sequential (the regression
    # at time n needs V at time n+1 which was just updated).
    for n in range(N - 1, 0, -1):
        itm = np.where(A[:, n] < K)[0]   # ITM for a put: A < K
        if len(itm) == 0:
            continue
        X      = design(S[itm, n], A[itm, n])
        Y      = np.exp(-r * dt * (N - n)) * V[itm]
        coef, *_ = np.linalg.lstsq(X, Y, rcond=None)
        cont   = X @ coef
        ex_now = exercise_payoff[itm, n]
        ex_mask = cont < ex_now
        V[itm[ex_mask]] = ex_now[ex_mask]

    price = np.exp(-r * T) * V.mean()
    return price

# Compare to:
# - European Asian put (no early exercise): ~3.39 from naive MC with
#   the same parameters. Averaging reduces the underlying variance,
#   making the Asian put cheaper than the vanilla European put.
# - American vanilla put (no averaging): ~5.81 from Listing 3.1
#   with these parameters.
# The American-Asian price should lie in the range [3.39, 5.81] ---
# cheaper than American vanilla (averaging reduces variance) but more
# expensive than European Asian (early exercise adds value).


In [ ]:

S0, K, r, sigma, T = 100, 100, 0.05, 0.20, 1.0
print(f"{'M':>8s} {'American-Asian put':>22s}")
for M in [5_000, 20_000, 50_000]:
    p = lsm_american_asian_put(S0, K, r, sigma, T, N=50, M=M, seed=42)
    print(f"{M:8d} {p:22.4f}")

In [ ]:
?range

In [ ]:

list(range(10, 0,-1))


In [ ]:
X=np.array([1.08, 1.07, 0.97, 0.77, 0.84])
Y=np.exp(-0.06)*np.array([0,0.07, 0.18, 0.20, 0.09])

In [ ]:
coeffs=np.polyfit(X,Y,2)

In [ ]:
def val_time_2(x):
    return np.sum(coeffs*[x**2, x, 1])
    

In [ ]:
[val_time_2(x)  for x in X ]

In [ ]:
np.array([2,4])*np.array([1,5])

In [ ]:
np.sum(coeffs*np.array([1.08**2,1.08,1]))


In [ ]:
?np.ones

In [ ]:
def power_basis_func (x, deg=3):
    return np.vstack([x**k for k in range(deg+1)]).T


In [ ]:
Xp=power_basis_func(X)


In [ ]:
A=np.vstack([X, np.ones(len(X))]).T

In [ ]:
m, c= np.linalg.lstsq(Xp,Y)

In [ ]:
?np.linalg.lstsq